In [9]:
# %pip install polars pyarrow

import datetime as dt
from pathlib import Path
import zipfile

import polars as pl

for zip_path in (Path("/content/data.zip"), Path("data.zip")):
    if zip_path.exists():
        extract_root = Path("/content/dbscoring")
        extract_root.mkdir(parents=True, exist_ok=True)
        if zipfile.is_zipfile(zip_path):
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(extract_root)
            print(f"Extracted {zip_path} to {extract_root}")
        else:
            print(f"Skip unpack: {zip_path} is not a valid zip archive")
        break

BASE_DIR = Path("../data/sources")
if not BASE_DIR.exists():
    BASE_DIR = Path("data/sources")

WAREHOUSE_ROOT = Path("../data/warehouse_polars")
if not WAREHOUSE_ROOT.exists():
    WAREHOUSE_ROOT = Path("data/warehouse_polars")

NOW_TS = dt.datetime.now().replace(microsecond=0)
START_TS = dt.datetime(2023, 1, 1, 0, 0, 0)
FUTURE_TS = dt.datetime(9999, 12, 31, 0, 0, 0)
SOURCE_REGISTRY = {
    "client_cards_daily": {
        "source_id": 1,
        "source_description": "Daily client card source.",
        "update_frequency": "daily",
        "target_table": "client_daily_attrs_scd2",
        "columns": (
            "srv_mb_nflag",
            "cc_stoplist",
            "lne_tot_debt_int_ovrd_rub_amt",
            "lne_tot_debt_ovrd_rub_amt",
        ),
    },
    "credit_cards_info": {
        "source_id": 2,
        "source_description": "Monthly credit card source.",
        "update_frequency": "monthly",
        "target_table": "client_monthly_attrs_scd1",
        "columns": (
            "client_income_amt",
            "oi_total_amt",
            "act_pl_os_rub_amt",
            "payroll_client_nflag",
            "inf_payroll_rub_amt",
            "legal_entity_amt",
            "inc_avg_risk_rub_amt",
            "otf_loan_rub_amt",
            "otf_fee_rub_amt",
            "inf_transfer_rub_amt",
            "cc_ever_nflag",
        ),
    },
    "deb_cards_info": {
        "source_id": 3,
        "source_description": "Monthly debit card source.",
        "update_frequency": "monthly",
        "target_table": "client_monthly_attrs_scd1",
        "columns": (
            "onl_bank_active_1m_nfalg",
            "auto_pay_active_qty",
            "cl_income_1m_amt",
            "dep_acc_1st_open_dt",
            "wdr_cash_6m_amt",
            "cash_op_6m_amt",
            "cash_3m_qty",
            "lst_balance_amt",
            "card_active_1m_nflag",
        ),
    },
}
ALL_COLUMNS = [col for meta in SOURCE_REGISTRY.values() for col in meta["columns"]]
ATTR_ID_BY_NAME = {name: idx for idx, name in enumerate(ALL_COLUMNS, start=1)}


In [10]:
# Physical schemas from the ER diagram.
DIM_SOURCES_SCHEMA = {
    "source_id": pl.Int32,
    "source_name": pl.String,
    "source_description": pl.String,
    "update_frequency": pl.String,
    "row_create_dtime": pl.Datetime,
    "row_update_dtime": pl.Datetime,
    "valid_from": pl.Datetime,
    "valid_to": pl.Datetime,
    "is_current": pl.Boolean,
}
DIM_ATTRIBUTES_SCHEMA = {
    "attribute_id": pl.Int32,
    "attribute_name": pl.String,
    "attribute_description": pl.String,
    "data_type": pl.String,
    "source_id": pl.Int32,
    "update_frequency": pl.String,
    "row_create_dtime": pl.Datetime,
    "row_update_dtime": pl.Datetime,
}
LOAD_LOG_SCHEMA = {
    "load_id": pl.Int64,
    "source_id": pl.Int32,
    "source_report_dt": pl.String,
    "load_start_dtime": pl.Datetime,
    "load_end_dtime": pl.Datetime,
    "target_table": pl.String,
    "load_status": pl.String,
    "rows_loaded": pl.Int64,
    "error_message": pl.String,
}
MONTHLY_SCHEMA = {
    "client_id": pl.Int32,
    "attribute_id": pl.Int32,
    "report_dt": pl.String,
    "attribute_value": pl.String,
    "source_id": pl.Int32,
    "row_update_dtime": pl.Datetime,
    "row_loading_id": pl.Int64,
    "row_hash_val": pl.String,
}
DAILY_SCHEMA = {
    "client_id": pl.Int32,
    "attribute_id": pl.Int32,
    "attribute_value": pl.String,
    "row_actual_from": pl.String,
    "row_actual_to": pl.String,
    "source_id": pl.Int32,
    "row_update_dtime": pl.Datetime,
    "row_loading_id": pl.Int64,
    "row_hash_val": pl.String,
}


In [11]:
# Read source slices.
def read_partition(path: Path) -> pl.DataFrame:
    files = sorted(path.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(path)
    return pl.read_parquet(files)

ccd_2023 = read_partition(BASE_DIR / "client_cards_daily" / "row_actual_to='2023-04-03'")
ccd_9999 = read_partition(BASE_DIR / "client_cards_daily" / "row_actual_to='9999-12-31'")
cci_02 = read_partition(BASE_DIR / "credit_cards_info" / "report_dt='2023-02-28'")
cci_03 = read_partition(BASE_DIR / "credit_cards_info" / "report_dt='2023-03-31'")
dci_02 = read_partition(BASE_DIR / "deb_cards_info" / "report_dt='2023-02-28'")
dci_03 = read_partition(BASE_DIR / "deb_cards_info" / "report_dt='2023-03-31'")

CLIENT_ID_MAP = {
    raw_id: idx
    for idx, raw_id in enumerate(
        sorted(
            set(pl.concat([
                ccd_2023.select(pl.col("id").cast(pl.String)),
                ccd_9999.select(pl.col("id").cast(pl.String)),
                cci_02.select(pl.col("id").cast(pl.String)),
                cci_03.select(pl.col("id").cast(pl.String)),
                dci_02.select(pl.col("id").cast(pl.String)),
                dci_03.select(pl.col("id").cast(pl.String)),
            ], how="vertical")["id"].to_list())
        ),
        start=1,
    )
}

def infer_data_type(name: str) -> str:
    if name.endswith("_dt") or name.endswith("_from") or name.endswith("_to"):
        return "DATE"
    if name.endswith("_amt"):
        return "DECIMAL"
    if name.endswith("_nflag") or name.endswith("_nfalg") or name.endswith("_qty"):
        return "INT"
    return "STRING"

def build_dim_sources() -> pl.DataFrame:
    rows = [{
        "source_id": meta["source_id"],
        "source_name": source_name,
        "source_description": meta["source_description"],
        "update_frequency": meta["update_frequency"],
        "row_create_dtime": NOW_TS,
        "row_update_dtime": NOW_TS,
        "valid_from": START_TS,
        "valid_to": FUTURE_TS,
        "is_current": True,
    } for source_name, meta in SOURCE_REGISTRY.items()]
    return pl.DataFrame(rows, schema=DIM_SOURCES_SCHEMA)

def build_dim_attributes() -> pl.DataFrame:
    rows = []
    for source_name, meta in SOURCE_REGISTRY.items():
        for column_name in meta["columns"]:
            rows.append({
                "attribute_id": ATTR_ID_BY_NAME[column_name],
                "attribute_name": column_name,
                "attribute_description": f"Attribute {column_name}",
                "data_type": infer_data_type(column_name),
                "source_id": meta["source_id"],
                "update_frequency": meta["update_frequency"],
                "row_create_dtime": NOW_TS,
                "row_update_dtime": NOW_TS,
            })
    return pl.DataFrame(rows, schema=DIM_ATTRIBUTES_SCHEMA)

def unpivot_table(df: pl.DataFrame, source_name: str) -> pl.DataFrame:
    meta = SOURCE_REGISTRY[source_name]
    prepared = df.with_columns([
        pl.col(column_name).cast(pl.String) for column_name in meta["columns"]
    ])
    melted = prepared.unpivot(
        index=["id", "row_update_dtime", "loading_id", "row_hash_val"] + (["report_dt"] if meta["update_frequency"] == "monthly" else ["row_actual_from", "row_actual_to"]),
        on=list(meta["columns"]),
        variable_name="attribute_name",
        value_name="attribute_value",
    ).with_columns(
        pl.col("id").cast(pl.String).replace_strict(CLIENT_ID_MAP).cast(pl.Int32).alias("client_id"),
        pl.col("attribute_name").replace_strict(ATTR_ID_BY_NAME).cast(pl.Int32).alias("attribute_id"),
        pl.col("attribute_value").cast(pl.String),
        pl.col("row_update_dtime").cast(pl.Datetime),
        pl.col("loading_id").cast(pl.Int64).alias("row_loading_id"),
        pl.lit(meta["source_id"]).cast(pl.Int32).alias("source_id"),
        pl.col("row_hash_val").cast(pl.String),
    )
    if meta["update_frequency"] == "monthly":
        return melted.select([
            pl.col("client_id"),
            pl.col("attribute_id"),
            pl.col("report_dt").cast(pl.String),
            pl.col("attribute_value"),
            pl.col("source_id"),
            pl.col("row_update_dtime"),
            pl.col("row_loading_id"),
            pl.col("row_hash_val"),
        ])
    return melted.select([
        pl.col("client_id"),
        pl.col("attribute_id"),
        pl.col("attribute_value"),
        pl.col("row_actual_from").cast(pl.String),
        pl.col("row_actual_to").cast(pl.String),
        pl.col("source_id"),
        pl.col("row_update_dtime"),
        pl.col("row_loading_id"),
        pl.col("row_hash_val"),
    ])


In [12]:
# Build final warehouse tables.
dim_sources = build_dim_sources()
dim_attributes = build_dim_attributes()

client_monthly_attrs_scd1 = pl.concat([
    unpivot_table(cci_02, "credit_cards_info"),
    unpivot_table(cci_03, "credit_cards_info"),
    unpivot_table(dci_02, "deb_cards_info"),
    unpivot_table(dci_03, "deb_cards_info"),
], how="vertical").unique(subset=["client_id", "attribute_id", "report_dt"], keep="last").cast(MONTHLY_SCHEMA)

client_daily_attrs_scd2 = pl.concat([
    unpivot_table(ccd_2023, "client_cards_daily"),
    unpivot_table(ccd_9999, "client_cards_daily"),
], how="vertical").unique(subset=["client_id", "attribute_id", "row_actual_from"], keep="last").cast(DAILY_SCHEMA)

load_log = pl.DataFrame([
    {"load_id": 1, "source_id": 1, "source_report_dt": "2023-04-03", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_daily_attrs_scd2", "load_status": "success", "rows_loaded": ccd_2023.height, "error_message": None},
    {"load_id": 2, "source_id": 1, "source_report_dt": "9999-12-31", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_daily_attrs_scd2", "load_status": "success", "rows_loaded": ccd_9999.height, "error_message": None},
    {"load_id": 3, "source_id": 2, "source_report_dt": "2023-02-28", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": cci_02.height, "error_message": None},
    {"load_id": 4, "source_id": 2, "source_report_dt": "2023-03-31", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": cci_03.height, "error_message": None},
    {"load_id": 5, "source_id": 3, "source_report_dt": "2023-02-28", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": dci_02.height, "error_message": None},
    {"load_id": 6, "source_id": 3, "source_report_dt": "2023-03-31", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": dci_03.height, "error_message": None},
], schema=LOAD_LOG_SCHEMA)

WAREHOUSE_ROOT.mkdir(parents=True, exist_ok=True)
dim_sources.write_parquet(WAREHOUSE_ROOT / "dim_sources.parquet")
dim_attributes.write_parquet(WAREHOUSE_ROOT / "dim_attributes.parquet")
load_log.write_parquet(WAREHOUSE_ROOT / "load_log.parquet")
client_monthly_attrs_scd1.write_parquet(WAREHOUSE_ROOT / "client_monthly_attrs_scd1.parquet")
client_daily_attrs_scd2.write_parquet(WAREHOUSE_ROOT / "client_daily_attrs_scd2.parquet")

{
    "dim_sources": dim_sources.height,
    "dim_attributes": dim_attributes.height,
    "load_log": load_log.height,
    "client_monthly_attrs_scd1": client_monthly_attrs_scd1.height,
    "client_daily_attrs_scd2": client_daily_attrs_scd2.height,
}


{'dim_sources': 3,
 'dim_attributes': 24,
 'load_log': 6,
 'client_monthly_attrs_scd1': 11441743,
 'client_daily_attrs_scd2': 761764}